In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import PROCESSED, RESULTS, savefig
from pypsa_helpers import solve_scenario, require_optimal, total_co2, total_system_cost, scale_capital_cost

import pypsa
import pandas as pd
import matplotlib.pyplot as plt

Sensitivity: H2 Fuel Cell Capital Cost (100% CO2 Reduction Scenario)

Mirrors `09_sensitivity_electrolysis_cost.ipynb` exactly, but scales the H2 **fuel cell**
(H2 -> electricity) capital cost instead of electrolysis (electricity -> H2). Comparing the
two sweeps side by side (see `14_h2_technology_comparison.ipynb`) answers which side of the
H2 conversion chain - storing energy or putting it back on the grid - is more cost-elastic.

Parameterized by the `FUEL_CELL_COST_FRACTION` environment variable (defaults to `1.0`), same
convention as the electrolysis sweep, so one notebook file covers all 5 levels.

In [ ]:
cost_fraction = float(os.environ.get("FUEL_CELL_COST_FRACTION", "1.0"))
assert 0.0 <= cost_fraction <= 1.0
print("fuel cell capital cost fraction:", cost_fraction)

In [ ]:
n = pypsa.Network(str(PROCESSED / "pypsa_network_base.nc"))
cost_scale = n.meta["cost_scale"]

n = scale_capital_cost(n, carrier="H2 fuel cell", fraction=cost_fraction)
n = solve_scenario(n, co2_limit=0)

In [ ]:
print("status:", n.meta["status"], "| termination condition:", n.meta["termination_condition"])

require_optimal(n)

total_cost_eur = total_system_cost(n, cost_scale=cost_scale)
total_co2_t = total_co2(n)
fuel_cell_capacity_mw = n.links.loc[n.links.carrier == "H2 fuel cell", "p_nom_opt"].sum()
electrolysis_capacity_mw = n.links.loc[n.links.carrier == "H2 electrolysis", "p_nom_opt"].sum()

print(f"fuel cell cost fraction:        {cost_fraction:.0%}")
print(f"total annual system cost:       {total_cost_eur:,.0f} EUR/yr")
print(f"total CO2 emissions:            {total_co2_t:,.0f} t/yr")
print(f"fuel cell capacity built:       {fuel_cell_capacity_mw:,.0f} MW")
print(f"electrolysis capacity built:    {electrolysis_capacity_mw:,.0f} MW")

In [ ]:
pct_label = f"{round(cost_fraction * 100)}pct"
n.export_to_netcdf(str(PROCESSED / f"pypsa_sensitivity_fuel_cell_{pct_label}.nc"))